## **Cleaning and EDA**

In [18]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import re
from build_dataset import DATA_DIR, FILE_ORDER

TARGET = "Label"

1. **Merge + Initial Filtering of Files**

The CIC-IDS2017 dataset captures network traffic data over a week (July 3-7 2017). Data is split into 8 csv files based on day of week and attack type, but was captured continuously with the same methodology, making merging valid.

In [19]:
# VERIFY SCHEMAS OF INDIVIDUAL DATA FILES
schemas = {}
for file in FILE_ORDER:
    cols = pd.read_csv(f"../data/{file}", nrows=0).columns.str.strip().tolist()
    cols = [c for c in cols if c != "Fwd Header Length.1"] # Normalize repeated header
    schemas[file] = tuple(cols)

unique_schemas = set(schemas.values())
print(f"{len(unique_schemas)} distinct schema(s) across data files.")


1 distinct schema(s) across data files.


In [20]:
# FILTER INDIVIDUAL DATA FILES AND MERGE
writer = None
schema_ref = None
MERGED_PATH = os.path.join(DATA_DIR, "merged_rawlabels.parquet")

for file in FILE_ORDER:
    path = os.path.join(DATA_DIR, file)
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()

    # Fix non-UTF8 encoding
    df[TARGET] = df[TARGET].astype(str).str.strip().str.replace("\ufffd", "-", regex=False)

    # Drop rows with missing and infinite values
    # TO-DO: REVISIT VALIDITY OF DROPPING SUCH ROWS
    feat_cols = [c for c in df.columns if c != TARGET]
    valid = np.ones(len(df), dtype=bool)
    for c in feat_cols:
        v = df[c].to_numpy()
        if np.issubdtype(v.dtype, np.number):
            valid &= np.isfinite(v.astype(np.float64, copy=False))
    df = df.loc[valid]

    # Drop within file duplicates
    # TO-DO: REVISIT VALIDITY OF DROPPING DUPLICATES
    df = df.drop_duplicates()

    # Reduce memory usage by using float32
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].astype(np.float32)

    # Display length of each source file
    df["Source File"] = file
    print(f"{file}: {len(df):,} rows")

    # Convert to Pyarrow Table and write into file
    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None: # First file only
        schema_ref = table.schema
        writer = pq.ParquetWriter(MERGED_PATH, schema_ref)
    else:
        table = table.cast(schema_ref)
    writer.write_table(table)
    del df, table

writer.close()
print("\nMerged to path:", MERGED_PATH)

Monday-WorkingHours.pcap_ISCX.csv: 502,650 rows
Tuesday-WorkingHours.pcap_ISCX.csv: 421,626 rows
Wednesday-workingHours.pcap_ISCX.csv: 610,492 rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 164,179 rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 252,790 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv: 184,044 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 213,777 rows
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 223,082 rows

Merged to path: c:\Users\lixin\portfolio\cybersecurity_anomaly_detection\processing\..\src\..\data\merged_rawlabels.parquet


In [21]:
# FILTERED MERGED DATA
pf = pq.ParquetFile(MERGED_PATH)
FINAL_PATH = os.path.join(DATA_DIR, "merge_complete.parquet")
all_cols = list(pf.schema_arrow.names)
feat_cols = [c for c in all_cols if c not in (TARGET, "source_file")]

# Find inter-file duplicates and constant columns
seen = set()
dup_flags = []
col_uniques = {c: set() for c in feat_cols}
for batch in pf.iter_batches(batch_size=250_000): # Work on data in batches
    # Checks for duplicates and inspects column values
    chunk = batch.to_pandas()
    h = pd.util.hash_pandas_object(chunk[feat_cols + [TARGET]], index=False).to_numpy()
    flags = np.array([(v in seen) or seen.add(v) for v in h], dtype=bool) # Checks if hash was seen
    dup_flags.append(flags)
    
    #Track unique values to find constant columns
    for c in feat_cols:
        if len(col_uniques[c]) <= 1:
            col_uniques[c].update(pd.unique(chunk[c])[:3].tolist())
dup_mask = np.concatenate(dup_flags)
constant_cols = [c for c, u in col_uniques.items() if len(u) <= 1]
keep_cols = [c for c in all_cols if c not in constant_cols] # Filters out constant columns
print(f"Number of inter-file duplicates: {dup_mask.sum()}")
print(f"Number of constant columns: {len(constant_cols)}")

# Write filtered file
writer = None
row_ptr = 0
rows_out = 0

# Restream file loading only kept columns
for batch in pq.ParquetFile(MERGED_PATH).iter_batches(batch_size=250_000, columns=keep_cols):
    chunk = batch.to_pandas()
    m = ~dup_mask[row_ptr: row_ptr + len(chunk)] # Boolean: Duplicate mask
    row_ptr += len(chunk)
    out = chunk.loc[m] # Keep only unique rows
    rows_out += len(out)

    # Write file
    table = pa.Table.from_pandas(out, preserve_index=False) # Convert to Pyarrow Table
    if writer is None:
        writer = pq.ParquetWriter(FINAL_PATH, table.schema)
    writer.write_table(table) 
    
writer.close()
print(f"\nWritten to path:", FINAL_PATH)

Number of inter-file duplicates: 205
Number of constant columns: 8

Written to path: c:\Users\lixin\portfolio\cybersecurity_anomaly_detection\processing\..\src\..\data\merge_complete.parquet


In [ ]:

def to_camel_case(col):
    # Split on any non-alphanumeric character
    parts = re.split(r"[^0-9a-zA-Z]+", col.strip())
    parts = [p for p in parts if p]  # remove empty strings
    return "".join(p.capitalize() for p in parts)

# Use function to convert all column names to CamelCase
table = pq.read_table(FINAL_PATH) 
new_schema = pa.schema([ 
    pa.field(to_camel_case(name), table.schema.field(name).type)
    for name in table.schema.names
])
new_table = table.rename_columns([to_camel_case(c) for c in table.schema.names])
print("New column names:", new_table.schema.names)

# Overwrite with column name changes
pq.write_table(new_table, FINAL_PATH)
print("Column names converted to CamelCase in:", FINAL_PATH)


New column names: ['Destinationport', 'Flowduration', 'Totalfwdpackets', 'Totalbackwardpackets', 'Totallengthoffwdpackets', 'Totallengthofbwdpackets', 'Fwdpacketlengthmax', 'Fwdpacketlengthmin', 'Fwdpacketlengthmean', 'Fwdpacketlengthstd', 'Bwdpacketlengthmax', 'Bwdpacketlengthmin', 'Bwdpacketlengthmean', 'Bwdpacketlengthstd', 'Flowbytess', 'Flowpacketss', 'Flowiatmean', 'Flowiatstd', 'Flowiatmax', 'Flowiatmin', 'Fwdiattotal', 'Fwdiatmean', 'Fwdiatstd', 'Fwdiatmax', 'Fwdiatmin', 'Bwdiattotal', 'Bwdiatmean', 'Bwdiatstd', 'Bwdiatmax', 'Bwdiatmin', 'Fwdpshflags', 'Fwdurgflags', 'Fwdheaderlength', 'Bwdheaderlength', 'Fwdpacketss', 'Bwdpacketss', 'Minpacketlength', 'Maxpacketlength', 'Packetlengthmean', 'Packetlengthstd', 'Packetlengthvariance', 'Finflagcount', 'Synflagcount', 'Rstflagcount', 'Pshflagcount', 'Ackflagcount', 'Urgflagcount', 'Cweflagcount', 'Eceflagcount', 'Downupratio', 'Averagepacketsize', 'Avgfwdsegmentsize', 'Avgbwdsegmentsize', 'Fwdheaderlength1', 'Subflowfwdpackets', 'S

#### **Working with Attack Types**

In [ ]:
# DETERMINE ATTACK TYPES APPEARING IN EACH FILE WITH COUNTS
rows = []
df = pq.read_table("../data/merge_complete.parquet", columns=["Label", "SourceFile"]).to_pandas()
for file in FILE_ORDER:
    labs = sorted(df.loc[df.SourceFile == file, "Label"].unique())
    attack_types = ", ".join(labs)
    rows.append({
        "file": file,
        "n_rows": int((df.SourceFile == file).sum()),
        "attack_types": attack_types
    })
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

ArrowInvalid: No match for FieldRef.Name(SourceFile) in Destinationport: float
Flowduration: float
Totalfwdpackets: float
Totalbackwardpackets: float
Totallengthoffwdpackets: float
Totallengthofbwdpackets: float
Fwdpacketlengthmax: float
Fwdpacketlengthmin: float
Fwdpacketlengthmean: float
Fwdpacketlengthstd: float
Bwdpacketlengthmax: float
Bwdpacketlengthmin: float
Bwdpacketlengthmean: float
Bwdpacketlengthstd: float
Flowbytess: float
Flowpacketss: float
Flowiatmean: float
Flowiatstd: float
Flowiatmax: float
Flowiatmin: float
Fwdiattotal: float
Fwdiatmean: float
Fwdiatstd: float
Fwdiatmax: float
Fwdiatmin: float
Bwdiattotal: float
Bwdiatmean: float
Bwdiatstd: float
Bwdiatmax: float
Bwdiatmin: float
Fwdpshflags: float
Fwdurgflags: float
Fwdheaderlength: float
Bwdheaderlength: float
Fwdpacketss: float
Bwdpacketss: float
Minpacketlength: float
Maxpacketlength: float
Packetlengthmean: float
Packetlengthstd: float
Packetlengthvariance: float
Finflagcount: float
Synflagcount: float
Rstflagc

In [ ]:
# DETERMINE FREQUENCY OF EACH LABEL
df = pq.read_table("../data/merge_complete.parquet", columns=["Label","SourceFile"]).to_pandas()
attack_counts = df["Label"].value_counts()

# Compute percentage
attack_percent = (attack_counts / len(df)) * 100

# Combine into a single dataframe
attack_summary = pd.DataFrame({
    "Count": attack_counts,
    "Percentage": attack_percent.round(2)   # round to 2 decimals
}).reset_index()
attack_summary



ArrowInvalid: No match for FieldRef.Name(source_file) in Destinationport: float
Flowduration: float
Totalfwdpackets: float
Totalbackwardpackets: float
Totallengthoffwdpackets: float
Totallengthofbwdpackets: float
Fwdpacketlengthmax: float
Fwdpacketlengthmin: float
Fwdpacketlengthmean: float
Fwdpacketlengthstd: float
Bwdpacketlengthmax: float
Bwdpacketlengthmin: float
Bwdpacketlengthmean: float
Bwdpacketlengthstd: float
Flowbytess: float
Flowpacketss: float
Flowiatmean: float
Flowiatstd: float
Flowiatmax: float
Flowiatmin: float
Fwdiattotal: float
Fwdiatmean: float
Fwdiatstd: float
Fwdiatmax: float
Fwdiatmin: float
Bwdiattotal: float
Bwdiatmean: float
Bwdiatstd: float
Bwdiatmax: float
Bwdiatmin: float
Fwdpshflags: float
Fwdurgflags: float
Fwdheaderlength: float
Bwdheaderlength: float
Fwdpacketss: float
Bwdpacketss: float
Minpacketlength: float
Maxpacketlength: float
Packetlengthmean: float
Packetlengthstd: float
Packetlengthvariance: float
Finflagcount: float
Synflagcount: float
Rstflag

Labels representing similar attacks are grouped together. This is because some classes are too small to evaluate on their own, and several labels represent the same type of attack. Infiltration and Heartbleed are excluded due to lack of data. 

1. DDoS
2. DoS
    DoS Hulk, DoS Golden Eye, DoS slowloris, DoS Slowhttptest
3. PortScan
4. Bot
5. BruteForce
    FTP-Patator, SSH-Patator
6. WebAttack
    Web Attack - Brute Force, Web Attack - XSS, Web Attack - Sql Injection
7. EXCLUDED
    Infiltration [36 attacks], Heartbleed [11 attacks]

In [ ]:
lmap = {
    "BENIGN": "BENIGN",
    "DDoS": "DDoS",
    "DoS Hulk": "DoS",
    "DoS GoldenEye": "DoS",
    "DoS slowloris": "DoS",
    "DoS Slowhttptest": "DoS",
    "PortScan": "PortScan",
    "Bot": "Bot",
    "FTP-Patator": "BruteForce",
    "SSH-Patator": "BruteForce",
    "Web Attack - Brute Force": "WebAttack",
    "Web Attack - XSS": "WebAttack",
    "Web Attack - Sql Injection": "WebAttack",
    "Infiltration": "EXCLUDED",
    "Heartbleed": "EXCLUDED",
}

df["LabelGroup"] = df["LabelGroup"].map(lmap)
print(df["LabelGroup"].value_counts(dropna=False))

NameError: name 'df' is not defined